# Case Study 1: E-Commerce AI Customer Support Agent

This notebook implements a RAG-based customer support agent for an e-commerce platform using:
- **LangChain** `create_agent` for agent orchestration and tool calling
- **FAISS** as a vector store for FAQ/knowledge base retrieval
- **Google Gemini** for intent classification, entity extraction, and response generation

The agent can:
1. Look up order status from a mock database
2. Answer policy questions using RAG over an FAQ knowledge base
3. Initiate returns for eligible orders
4. Escalate to a human agent when the customer is frustrated or the issue is complex

In [1]:
import os
from datetime import datetime, timedelta

from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages import HumanMessage


import os
os.environ["GOOGLE_API_KEY"] = "your-gemini-api-key-here"

print("Dependencies loaded.")
def _clean(content):
    """Extract plain text from Gemini response (handles str or list-of-dicts with extras/signature)."""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return "".join(b["text"] for b in content if isinstance(b, dict) and "text" in b)
    return str(content)



Dependencies loaded.


In [2]:
# Mock order database
ORDERS = {
    "ORD-1234": {
        "order_id": "ORD-1234", "customer_id": "CUST-001", "status": "shipped",
        "items": ["Wireless Headphones", "Phone Case"],
        "order_date": "2024-01-10", "delivery_date": "2024-01-20", "total": 89.99,
    },
    "ORD-5678": {
        "order_id": "ORD-5678", "customer_id": "CUST-002", "status": "delivered",
        "items": ["Running Shoes"],
        "order_date": "2024-01-05", "delivery_date": "2024-01-10", "total": 129.99,
    },
    "ORD-9012": {
        "order_id": "ORD-9012", "customer_id": "CUST-003", "status": "processing",
        "items": ["Laptop Stand", "USB-C Hub", "Webcam"],
        "order_date": "2024-01-18", "delivery_date": "2024-01-28", "total": 214.97,
    },
    "ORD-3456": {
        "order_id": "ORD-3456", "customer_id": "CUST-004", "status": "delivered",
        "items": ["Winter Jacket"],
        "order_date": "2023-12-15", "delivery_date": "2023-12-22", "total": 199.99,
    },
}

print(f"Loaded {len(ORDERS)} mock orders: {list(ORDERS.keys())}")

Loaded 4 mock orders: ['ORD-1234', 'ORD-5678', 'ORD-9012', 'ORD-3456']


In [3]:
# FAQ knowledge base
FAQ_DOCUMENTS = [
    "Return Policy: Items can be returned within 30 days of delivery for a full refund. Items must be in original condition with tags attached. Electronics must include all original accessories and packaging.",
    "Refund Process: Once we receive your returned item and verify its condition, refunds are processed within 5-7 business days. Refunds are issued to the original payment method used at checkout.",
    "Shipping Times: Standard shipping takes 5-7 business days. Express shipping takes 2-3 business days. Overnight shipping is available for an additional fee and delivers next business day by 5 PM.",
    "Free Shipping: Orders over $50 qualify for free standard shipping. Free shipping does not apply to oversized items or international orders.",
    "Damaged Items: If your item arrived damaged, please contact us within 48 hours of delivery with photos. We will send a replacement or issue a full refund at no extra cost — no return shipment needed.",
    "Order Cancellation: Orders can be cancelled within 1 hour of placement if they haven't entered the processing stage. Once processing begins, the order cannot be cancelled but can be returned after delivery.",
    "International Shipping: We ship to over 50 countries. International orders may be subject to customs duties and taxes which are the responsibility of the buyer. Delivery takes 10-21 business days.",
    "Gift Returns: Gift recipients can return items for store credit. A gift receipt or the order number from the purchaser is required. Store credit does not expire.",
    "Price Match: We offer price matching within 14 days of purchase. If you find the same item at a lower price from an authorized retailer, contact us with proof and we will refund the difference.",
    "Account & Privacy: Your personal data is protected under our privacy policy. We never share your information with third parties for marketing. You can delete your account at any time from account settings.",
]

print(f"Knowledge base loaded with {len(FAQ_DOCUMENTS)} FAQ documents.")

Knowledge base loaded with 10 FAQ documents.


In [4]:
# Simple keyword-based knowledge search (no external embedding API needed)
from difflib import SequenceMatcher

def search_faq(query: str, k: int = 3) -> list[str]:
    """Search FAQ documents by keyword relevance."""
    query_lower = query.lower()
    scored = []
    for doc in FAQ_DOCUMENTS:
        score = sum(1 for word in query_lower.split() if word in doc.lower())
        score += SequenceMatcher(None, query_lower, doc.lower()).ratio()
        scored.append((score, doc))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [doc for _, doc in scored[:k]]

print(f"Knowledge base search ready with {len(FAQ_DOCUMENTS)} FAQ documents.")

Knowledge base search ready with 10 FAQ documents.


In [5]:
# Define tools

@tool
def lookup_order(order_id: str) -> str:
    """Look up an order by its order ID (e.g. ORD-1234). Returns order status, items, and delivery date."""
    order = ORDERS.get(order_id.strip().upper())
    if not order:
        return f"No order found with ID '{order_id}'. Please verify the order ID and try again."
    return (
        f"Order {order['order_id']}:\n"
        f"  Status: {order['status']}\n"
        f"  Items: {', '.join(order['items'])}\n"
        f"  Order Date: {order['order_date']}\n"
        f"  Expected Delivery: {order['delivery_date']}\n"
        f"  Total: ${order['total']:.2f}"
    )


@tool
def initiate_return(order_id: str) -> str:
    """Initiate a return for an order. Only works for delivered orders within the return window."""
    order = ORDERS.get(order_id.strip().upper())
    if not order:
        return f"No order found with ID '{order_id}'."
    if order["status"] != "delivered":
        return f"Order {order_id} has status '{order['status']}' and cannot be returned yet. Only delivered orders can be returned."
    order["status"] = "return_initiated"
    return (
        f"Return initiated for order {order_id} ({', '.join(order['items'])}).\n"
        f"A prepaid shipping label will be emailed within 24 hours.\n"
        f"Refund of ${order['total']:.2f} will be processed within 5-7 business days after we receive the item."
    )


@tool
def search_knowledge_base(query: str) -> str:
    """Search the company FAQ and policy knowledge base. Use this for questions about return policies, shipping, refunds, account issues, and general company policies."""
    results = search_faq(query)
    return "\n\n".join(results)

tools = [lookup_order, initiate_return, search_knowledge_base]
print(f"Registered {len(tools)} tools: {[t.name for t in tools]}")

Registered 3 tools: ['lookup_order', 'initiate_return', 'search_knowledge_base']


In [6]:
# Create the agent using LangChain's create_agent API

SYSTEM_PROMPT = """You are a friendly and professional customer support agent for ShopEasy, 
an online e-commerce store. Your role is to help customers with their orders, returns, 
refunds, and general questions.

Guidelines:
- Always be empathetic, patient, and professional.
- Use the lookup_order tool when the customer provides or mentions an order ID.
- Use the search_knowledge_base tool to answer policy and general questions.
- Use the initiate_return tool only when the customer explicitly confirms they want to return.
- If the customer is very frustrated, angry, mentions legal action, or asks for a manager,
  escalate by saying: "I'm escalating your case to a senior specialist who will contact you
  within 1 hour." Include [ESCALATED] in your response.
- Never make up order information — always use the lookup tool.
- Keep responses concise but helpful.
"""

agent = create_agent(
    "google_genai:gemini-2.5-flash",
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
)

print("Agent created successfully.")

Agent created successfully.


In [7]:
# Helper function to run queries through the agent

def _clean(content):
    if isinstance(content, list):
        return "".join(b["text"] for b in content if isinstance(b, dict) and "text" in b)
    return str(content) if content else ""

def run_conversation(queries: list[str]):
    """Run a list of customer queries through the agent and print results."""
    for i, query in enumerate(queries, 1):
        print(f"\n{'='*70}")
        print(f"QUERY {i}: {query}")
        print(f"{'='*70}")
        result = agent.invoke({"messages": [HumanMessage(content=query)]})
        response = _clean(result["messages"][-1].content)
        print(f"\nAGENT RESPONSE:\n{response}\n")

In [8]:
# Demo — run sample customer queries

sample_queries = [
    "Where is my order ORD-1234?",
    "I want to return the shoes I bought. My order is ORD-5678.",
    "What's your refund policy?",
    "I've been waiting 2 weeks and no one is helping me. This is completely unacceptable and I want to speak to a manager immediately!",
]

run_conversation(sample_queries)


QUERY 1: Where is my order ORD-1234?

AGENT RESPONSE:
Your order ORD-1234 has been shipped and is expected to be delivered by January 20, 2024. It includes Wireless Headphones and a Phone Case.


QUERY 2: I want to return the shoes I bought. My order is ORD-5678.

AGENT RESPONSE:
I've initiated the return for your order ORD-5678. A prepaid shipping label will be emailed to you within 24 hours. Once we receive the shoes, a refund of $129.99 will be processed within 5-7 business days.


QUERY 3: What's your refund policy?

AGENT RESPONSE:
Our refund policy states that items can be returned within 30 days of delivery for a full refund. They must be in original condition with tags attached. For electronics, all original accessories and packaging must be included. Once we receive and verify the condition of your returned item, refunds are processed within 5-7 business days to the original payment method. We also offer price matching within 14 days of purchase.


QUERY 4: I've been waiting 